**Cell #01**

# RAG11 Nutrition — Stage 6: Multi-Query / Question Splitting Examples

Adds **multi-query / question splitting** on top of the Stage 2 retrieval +
generation pipeline: *before* any embedding or keyword search happens,
Claude looks at the question and decides whether it secretly bundles more
than one independent information need — a comparison, a list of distinct
nutrients/populations/conditions in one sentence. If it does, the question
is split into self-contained sub-questions, each one is searched
separately, and every resulting ranked list is fused back into one with the
same Reciprocal Rank Fusion `stage2_ask_examples3_hybrid_search.ipynb`
already uses for dense + keyword search.

**The problem in plain terms:** one search query can only point in one
"direction." A question that's really two questions glued together
confuses a single vector (or keyword) search.

Example: *"How does soluble fiber's effect on LDL cholesterol differ from
insoluble fiber's effect?"*

- This is secretly two questions: (a) "What does soluble fiber do to LDL
  cholesterol?" and (b) "What does insoluble fiber do to LDL cholesterol?"
- A single embedding of the whole question ends up as a blurry average of
  both topics, which can under-match either one.
- The fix: have Claude split it into the two sub-questions, run retrieval
  separately for each, then combine both result sets before reranking. Now
  you're guaranteed chunks about both fiber types show up, instead of
  hoping one combined search happens to surface both.

See `reusable_code/multi_query_question_splitting.py` for the
implementation, and
`documentation/HOW_IT_WORKS_Multi_Query_Question_Splitting.html` for the
full write-up (including how this composes with hybrid search, HyDE,
reranking, and parent-chunk expansion).

In [1]:
# Cell #02
from reusable_code import (
    init_clients,
    ask_question,
    retrieve_chunks,
    retrieve_chunks_multi_query,
    split_into_subquestions,
    hybrid_search,
    NUM_CONTEXT_CHUNKS,
    MAX_SUBQUESTIONS,
    EMBEDDING_MODEL,
    GENERATION_MODEL,
)

clients = init_clients()
print("Clients ready.")
print(f"EMBEDDING_MODEL={EMBEDDING_MODEL!r}  GENERATION_MODEL={GENERATION_MODEL!r}  MAX_SUBQUESTIONS={MAX_SUBQUESTIONS}")

Clients ready.
EMBEDDING_MODEL='voyage-3'  GENERATION_MODEL='claude-sonnet-5'  MAX_SUBQUESTIONS=4


**Cell #03**

## A helper to see "one combined search" vs. "multi-query split" side by side

`show_split_comparison()` first calls `split_into_subquestions()` directly
so the split decision itself is visible, then retrieves the same question
two ways — plain `retrieve_chunks()` (today's baseline: one combined
search) vs. `retrieve_chunks_multi_query()` (Claude splits first, *if* it
judges the question compound) — and prints both result sets side by side,
flagging any chunk multi-query surfaced that the baseline search missed
entirely.

In [2]:
# Cell #04
def show_split_comparison(question: str, match_count: int = NUM_CONTEXT_CHUNKS):
    """Split `question`, retrieve it both the old way (retrieve_chunks())
    and the multi-query way (retrieve_chunks_multi_query()), and print both
    side by side. Returns (subquestions, baseline_chunks, multi_query_chunks)
    so the caller can inspect or reuse any of them."""
    print(f"Q: {question}\n")

    subquestions = split_into_subquestions(question)
    if len(subquestions) > 1:
        print(f"split_into_subquestions() -> split into {len(subquestions)} sub-questions:")
    else:
        print("split_into_subquestions() -> judged already atomic, left unchanged:")
    for i, sq in enumerate(subquestions, start=1):
        print(f"  {i}. {sq}")
    print()

    baseline = retrieve_chunks(question, match_count=match_count)
    multi = retrieve_chunks_multi_query(question, match_count=match_count)

    baseline_guids = {row["rowGUID"] for row in baseline}
    multi_guids = {row["rowGUID"] for row in multi}
    only_multi = multi_guids - baseline_guids

    print(f"Baseline retrieve_chunks() -- {len(baseline)} chunk(s):")
    for row in baseline:
        snippet = row["rowJSON"]["text"].replace("\n", " ")[:110]
        print(f"  - {snippet}...")

    print(f"\nMulti-query retrieve_chunks_multi_query() -- {len(multi)} chunk(s):")
    for row in multi:
        snippet = row["rowJSON"]["text"].replace("\n", " ")[:110]
        marker = "  <-- NOT in the baseline search" if row["rowGUID"] in only_multi else ""
        print(f"  - {snippet}...{marker}")

    print(f"\n{len(only_multi)} chunk(s) surfaced by multi-query that the baseline combined search missed.")
    return subquestions, baseline, multi


example_results = {}

**Cell #05**

## Three nutrition examples for the multi-query demonstration

Each chosen for a different reason:

1. **The running comparison example** — "soluble vs. insoluble fiber and
   LDL cholesterol." Secretly two independent asks glued together with
   "differ from"; the canonical case multi-query splitting exists for.
2. **A compound question mixing two related-but-distinct asks** — an RDA
   lookup plus a dependent mechanism, joined by "and." Tests whether
   splitting correctly separates two genuinely different information needs
   even when they share a nutrient.
3. **An atomic control question** — long and detailed, but a single ask.
   This tests the *other* half of the mechanism: that `split_into_subquestions()`
   correctly leaves a non-compound question unchanged instead of
   over-splitting it into meaningless fragments.

In [3]:
# Cell #06
EXAMPLE_QUESTIONS = [
    # 1. The running comparison example -- both halves glued together with
    #    "differ from"; the canonical multi-query case.
    "How does soluble fiber's effect on LDL cholesterol differ from insoluble fiber's effect?",
    # 2. Two related-but-distinct asks joined by "and" -- an RDA lookup plus
    #    a dependent mechanism that shares a nutrient (vitamin D) but is its
    #    own information need (calcium absorption).
    "What is the RDA for vitamin D, and how does calcium absorption depend on it?",
    # 3. Atomic control: long and specific, but a single ask -- should NOT
    #    be split, demonstrating the mechanism doesn't over-split.
    "What is the recommended daily sodium intake for adults with hypertension according to clinical nutrition guidelines?",
]
for i, q in enumerate(EXAMPLE_QUESTIONS, start=1):
    print(f"{i}. {q}")

1. How does soluble fiber's effect on LDL cholesterol differ from insoluble fiber's effect?
2. What is the RDA for vitamin D, and how does calcium absorption depend on it?
3. What is the recommended daily sodium intake for adults with hypertension according to clinical nutrition guidelines?


**Cell #07**

### Example 1 — the running comparison (both halves should surface)

In [4]:
# Cell #08
example_results[EXAMPLE_QUESTIONS[0]] = show_split_comparison(EXAMPLE_QUESTIONS[0])

Q: How does soluble fiber's effect on LDL cholesterol differ from insoluble fiber's effect?

split_into_subquestions() -> split into 2 sub-questions:
  1. What effect does soluble fiber have on LDL cholesterol?
  2. What effect does insoluble fiber have on LDL cholesterol?

Baseline retrieve_chunks() -- 5 chunk(s):
  - [Source: Advanced_Nutrition_and_Human_Metabolism.pdf | Section: Ch 4: Fiber | Pages 23-28]  beneficial for low...
  - [Source: Advanced_Nutrition_and_Human_Metabolism.pdf | Section: Ch 4: Fiber | Pages 23-28]  Chapter 4 Summary ...
  - [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.pdf | Section: Eating to Stay H...
  - [Source: Advanced_Nutrition_and_Human_Metabolism.pdf | Section: Best Quotes (by chapter, with page numbers) | ...
  - [Source: _OceanofPDF.com_Encyclopedia_of_foods_-_Mayo_Clinic.pdf | Section: Chapter 2. The Nutrients and Other...

Multi-query retrieve_chunks_multi_query() -- 5 chunk(s):
  - [Source: Advanced_Nutrition_and

**Cell #09**

### Example 2 — two related-but-distinct asks joined by "and"

In [5]:
# Cell #10
example_results[EXAMPLE_QUESTIONS[1]] = show_split_comparison(EXAMPLE_QUESTIONS[1])

Q: What is the RDA for vitamin D, and how does calcium absorption depend on it?

split_into_subquestions() -> split into 2 sub-questions:
  1. What is the RDA for vitamin D?
  2. How does calcium absorption depend on vitamin D?

Baseline retrieve_chunks() -- 5 chunk(s):
  - [Source: vdoc.pub_medical-nutrition-and-disease-a-case-based-approach.pdf | Section: 2: Vitamins, Minerals, an...
  - [Source: _OceanofPDF.com_Encyclopedia_of_foods_-_Mayo_Clinic.pdf | Section: Chapter 3. The Food-Health Connect...
  - [Source: vdoc.pub_medical-nutrition-and-disease-a-case-based-approach.pdf | Section: 2: Vitamins, Minerals, an...
  - [Source: vdoc.pub_medical-nutrition-and-disease-a-case-based-approach.pdf | Section: 2: Vitamins, Minerals, an...
  - [Source: vdoc.pub_medical-nutrition-and-disease-a-case-based-approach.pdf | Section: 2: Vitamins, Minerals, an...

Multi-query retrieve_chunks_multi_query() -- 5 chunk(s):
  - [Source: vdoc.pub_medical-nutrition-and-disease-a-case-based-approach.pdf | S

**Cell #11**

### Example 3 — atomic control (should NOT be split)

In [6]:
# Cell #12
example_results[EXAMPLE_QUESTIONS[2]] = show_split_comparison(EXAMPLE_QUESTIONS[2])

Q: What is the recommended daily sodium intake for adults with hypertension according to clinical nutrition guidelines?

split_into_subquestions() -> judged already atomic, left unchanged:
  1. What is the recommended daily sodium intake for adults with hypertension according to clinical nutrition guidelines?

Baseline retrieve_chunks() -- 5 chunk(s):
  - [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.pdf | Section: Eating to Stay H...
  - [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.pdf | Section: Eating to Stay H...
  - [Source: _OceanofPDF.com_Encyclopedia_of_foods_-_Mayo_Clinic.pdf | Section: Chapter 3. The Food-Health Connect...
  - [Source: vdoc.pub_medical-nutrition-and-disease-a-case-based-approach.pdf | Section: 2: Vitamins, Minerals, an...
  - [Source: vdoc.pub_medical-nutrition-and-disease-a-case-based-approach.pdf | Section: 2: Vitamins, Minerals, an...

Multi-query retrieve_chunks_multi_query() -- 5 chunk(s)

**Cell #13**

## `split_into_subquestions()` in isolation — the parsing and fallback rules

`split_into_subquestions()` is a small, focused function: ask Claude, parse
its `"SUBQ: "`-prefixed lines, and fall back to `[question]` unchanged if
nothing parses. Worth seeing directly against a few more questions,
including one that's short enough Claude might phrase its "no split"
answer differently, to confirm the fallback behavior holds up.

In [7]:
# Cell #14
isolation_questions = [
    "How does soluble fiber's effect on LDL cholesterol differ from insoluble fiber's effect?",
    "What is the RDA for vitamin D, and how does calcium absorption depend on it?",
    "Is vitamin C water-soluble?",
    "What are the best food sources of omega-3 fatty acids, and how do EPA and DHA differ in their cardiovascular effects?",
]
for q in isolation_questions:
    subqs = split_into_subquestions(q)
    label = "split" if len(subqs) > 1 else "unchanged"
    print(f"Q: {q}")
    print(f"  -> {label} ({len(subqs)} sub-question(s)):")
    for sq in subqs:
        print(f"     - {sq}")
    print()

Q: How does soluble fiber's effect on LDL cholesterol differ from insoluble fiber's effect?
  -> split (2 sub-question(s)):
     - What is soluble fiber's effect on LDL cholesterol?
     - What is insoluble fiber's effect on LDL cholesterol?

Q: What is the RDA for vitamin D, and how does calcium absorption depend on it?
  -> split (2 sub-question(s)):
     - What is the RDA for vitamin D?
     - How does calcium absorption depend on vitamin D?

Q: Is vitamin C water-soluble?
  -> unchanged (1 sub-question(s)):
     - Is vitamin C water-soluble?

Q: What are the best food sources of omega-3 fatty acids, and how do EPA and DHA differ in their cardiovascular effects?
  -> split (2 sub-question(s)):
     - What are the best food sources of omega-3 fatty acids?
     - How do EPA and DHA differ in their cardiovascular effects?



**Cell #15**

## `ask_question(..., use_multi_query=...)` end to end

`use_multi_query` is an optional keyword argument on `ask_question` — it
defaults to `False`, so every existing call in
`stage2_ask_examples1/2/3/4/5.ipynb` behaves exactly as before. Passing
`use_multi_query=True` runs splitting **first**, before `use_hybrid`/`use_hyde`
pick a retrieval method for each sub-question; `use_rerank` and
`expand_to_parents` still run afterward, on whatever the fused,
multi-query-aware candidate pool contains.

In [8]:
# Cell #16
demo_question = EXAMPLE_QUESTIONS[0]

baseline_answer = ask_question(
    demo_question, match_count=NUM_CONTEXT_CHUNKS,
    use_multi_query=False, use_hybrid=False, use_hyde=False, expand_to_parents=False,
)
multi_query_answer = ask_question(
    demo_question, match_count=NUM_CONTEXT_CHUNKS, use_multi_query=True,
    use_hybrid=False, use_hyde=False, expand_to_parents=False,
)
full_pipeline_answer = ask_question(
    demo_question, match_count=NUM_CONTEXT_CHUNKS,
    use_multi_query=True, use_hybrid=True, use_rerank=True, expand_to_parents=True,
    use_hyde=False,
)

print("=== Baseline (no multi-query) ===")
print(f"Chunks used: {baseline_answer['chunks_used']}   Source pages: {baseline_answer['source_pages']}")
print(baseline_answer["answer"][:600])

print("\n=== Multi-query ===")
print(f"Sub-questions searched: {multi_query_answer['subquestions']}")
print(f"Chunks used: {multi_query_answer['chunks_used']}   Source pages: {multi_query_answer['source_pages']}")
print(multi_query_answer["answer"][:600])

print("\n=== Full pipeline (multi-query + hybrid + rerank + parent expansion) ===")
print(f"Chunks used: {full_pipeline_answer['chunks_used']}   Source pages: {full_pipeline_answer['source_pages']}")
print(full_pipeline_answer["answer"][:600])

=== Baseline (no multi-query) ===
Chunks used: 5   Source pages: [23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181]
Based on the excerpts, soluble and insoluble fiber have distinct roles, and only soluble fiber is directly linked to lowering LDL cholesterol.

- **Soluble fiber**: Excerpt 1 states that soluble fiber is "beneficial for lowering cholesterol and

**Cell #17**

## Summary across the 3 examples

Same idea as the hybrid-search and parent-expansion notebooks' summary
tables: a quick scan of how many sub-questions each example split into, how
many chunks each retrieval mode returned, and how much multi-query added
that the baseline combined search missed entirely.

In [9]:
# Cell #18
print(f"{'#':<3} {'sub-Qs':<7} {'baseline':<9} {'multi-query':<12} {'only-multi':<11} question")
for i, question in enumerate(EXAMPLE_QUESTIONS, start=1):
    subqs, baseline, multi = example_results[question]
    baseline_guids = {r["rowGUID"] for r in baseline}
    multi_guids = {r["rowGUID"] for r in multi}
    only_multi = len(multi_guids - baseline_guids)
    short_q = question if len(question) <= 55 else question[:52] + "..."
    print(f"{i:<3} {len(subqs):<7} {len(baseline):<9} {len(multi):<12} {only_multi:<11} {short_q}")

#   sub-Qs  baseline  multi-query  only-multi  question
1   2       5         5            0           How does soluble fiber's effect on LDL cholesterol d...
2   2       5         5            2           What is the RDA for vitamin D, and how does calcium ...
3   1       5         5            0           What is the recommended daily sodium intake for adul...


**Cell #19**

## For stakeholders — what this means in plain terms

| Without multi-query | With multi-query |
| --- | --- |
| A comparison question ("X vs. Y") is searched as one blurry combined query. | Each side of the comparison gets its own precise, independent search. |
| Whichever topic the sentence happens to emphasize can dominate retrieval, burying the other. | Both topics are guaranteed a fair search before their results are combined. |
| No way to tell, after the fact, *why* a chunk was (or wasn't) retrieved. | The sub-questions actually searched are returned alongside the answer (`result["subquestions"]`), so it's auditable. |
| Nothing extra for an already-simple question. | Same: an atomic question costs one extra (cheap) "should I split this?" call, then comes back unchanged — negligible overhead for the common case. |

**Net effect:** more complete, better-grounded answers on comparison and
multi-part questions, at the cost of one extra Claude call (to decide
whether to split) plus one extra retrieval call per sub-question when a
split does happen — a good trade for a clinical nutrition Q&A system,
where silently answering only half of a comparison is worse than a
slightly slower answer.

## For AI engineers — what this means technically

- **Zero schema change.** Splitting reads nothing from and writes nothing
  to Supabase — it's a pure Claude call
  (`reusable_code/multi_query_question_splitting.py`) that decides how many
  times to call the *existing* retrieval RPCs, exactly like HyDE only
  changes *what text* gets embedded, not the retrieval mechanism itself.
- **Composable, not a replacement.** It runs *first* — before `use_hybrid`/
  `use_hyde` pick a per-sub-question retrieval method, and before
  `use_rerank`/`expand_to_parents` narrow and expand the result — see
  `ask_question(use_multi_query=..., use_hybrid=..., use_rerank=...,
  expand_to_parents=...)` in Cell #16 above.
- **Fusion is reused, not reinvented.** The exact same
  `reciprocal_rank_fusion()` that merges dense + keyword search in
  `hybrid_search.py` merges "one ranked list per sub-question" here — see
  `retrieve_chunks_multi_query()` in
  `reusable_code/multi_query_question_splitting.py`.
- **Graceful degradation.** If Claude's split response can't be parsed at
  all, `split_into_subquestions()` falls back to `[question]` unchanged —
  the caller always has something to retrieve on; never zero sub-questions.
- **Bounded, not unbounded.** `max_subquestions` (default
  `MAX_SUBQUESTIONS = 4`) caps how many sub-questions — and therefore how
  many extra retrieval calls — one question can trigger, so a pathological
  or adversarial question can't fan out into an unbounded number of calls.
- **Ignores HyDE, composes with hybrid search.** Same reasoning `use_hybrid`
  already uses to ignore `use_hyde`: multi-query needs a single-question
  retrieval function per sub-question, and each sub-question is itself
  searched with `hybrid_search()` when `use_hybrid=True` — see Cell #3's
  helper and `documentation/HOW_IT_WORKS_Multi_Query_Question_Splitting.html`.

**Cell #20**

## Save workspace to GitHub

Synchronize this notebook and any code changes to GitHub (with auto lock
recovery and conflict resolution), the same helper
`stage2_ask_examples5_hypothetical_document_embedding.ipynb` uses -- shared
via `reusable_code.save_to_github`.

In [10]:
# Cell #21
from reusable_code import save_to_github

save_to_github("stage2_ask_examples6_multi_query_question_splitting.ipynb - multi-query/question-splitting examples added")

  RAG11 -> GitHub Robust Sync Utility
  Directory : /Users/mgtimber/CV26/RAG11
  Remote URL: https://github.com/fotomain/RAG11-nutriciology.git
[0/5] Checking repository health & clearing stale locks...
[1/5] Git repository verified.
[2/5] Origin remote verified: https://github.com/fotomain/RAG11-nutriciology.git
[3/5] Staging workspace files...
[4/5] Committing changes: "stage2_ask_examples6_multi_query_question_splitting.ipynb - multi-query/question-splitting examples added"
[main 539efe8] stage2_ask_examples6_multi_query_question_splitting.ipynb - multi-query/question-splitting examples added
 2 files changed, 1103 insertions(+)
 create mode 100644 documentation/Project_WBS_LRM_Build_Long_Reasonig_Model.html
 create mode 100644 stage2_ask_examples6_multi_query_question_splitting.ipynb
[5/5] Synchronizing with GitHub (main)...
      Pushing to origin main (attempt 1/3)...
branch 'main' set up to track 'origin/main'.

  Successfully synchronized with GitHub!
  Branch    : main
  Commi

To https://github.com/fotomain/RAG11-nutriciology.git
   fcfceb5..539efe8  main -> main



True